In [1]:
from quri_parts.qsub.lib import std
from quri_parts.qsub.opsub import OpSubDef, opsub
from quri_parts.qsub.resolve import SubRepository, default_repository, resolve_sub
from quri_parts.qsub.sub import SubBuilder
from quri_parts.qsub.op import Op
from quri_parts.qsub.sub import Sub

class _FFF(OpSubDef):
    name = "FFF"
    qubit_count = 1
    self_inverse = True

    def sub(self, builder: SubBuilder) -> None:
        builder.add_op(std.X, builder.qubits)

FFF, _ = opsub(_FFF)


def alt_fff_subresolver(op: Op, repo: SubRepository) -> Sub:
    builder = SubBuilder(1)
    builder.add_op(std.SqrtX, builder.qubits)
    return builder.build()


def alt_inverse_fff_subresolver(op: Op, repo: SubRepository) -> Sub:
    inner_op = op.id.params[0]
    assert isinstance(inner_op, Op)
    sub = alt_fff_subresolver(inner_op, repo)
    return std.inverse.get_inverted_sub(sub)



addition_repo = SubRepository()
addition_repo.register_sub_resolver(FFF, alt_fff_subresolver)
addition_repo.register_sub_resolver(
    std.Inverse, 
    alt_inverse_fff_subresolver, 
    std.inverse.inverse_target_condition(FFF)
)
addition_repo

In [3]:
from quri_parts.qsub.compile import compile_sub
from quri_parts.qsub.eval import GateCountEvaluatorHooks
from quri_parts.qsub.evaluate import Evaluator
from quri_parts.qsub.primitive import AllBasicSet

gate_counter = Evaluator(GateCountEvaluatorHooks())
new_repo = default_repository().chain([addition_repo])


builder = SubBuilder(1)
builder.add_op(FFF, builder.qubits)
builder.add_op(std.Inverse(FFF), builder.qubits)

with new_repo:
    msub = compile_sub(builder.build(), AllBasicSet, new_repo)
    gate_count = gate_counter.run(msub)

{g[1]: count for g, count in gate_count.items()}

{'SqrtX': 1, 'SqrtXdag': 1}